# Pre-Emphasis and De-Emphasis

This notebook extracts the pre/de-emphasis comparison from the legacy audio demo. It shows how FM systems tilt the audio spectrum before transmission and undo that tilt after demodulation to suppress high-frequency hiss.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Spectral Tilt as a Noise Countermeasure

FM demodulated noise rises with frequency. Pre-emphasis boosts high audio frequencies before transmission, and de-emphasis cuts them back down after reception, which also knocks down the extra hiss.

In [ ]:
message = signal.sosfilt(signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"), voice_work)
message = normalize(message)
carrier_freq = 20_000
freq_dev = 2_500

pre = normalize(pre_emphasis(message, WORK_FS))
fm_plain = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=freq_dev)
fm_emph = fm_modulate(pre, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=freq_dev)

fm_plain_noisy, _ = add_awgn(fm_plain, 10, seed=42)
fm_emph_noisy, _ = add_awgn(fm_emph, 10, seed=42)

plain_demod = fm_demodulate(fm_plain_noisy, fs=WORK_FS)
emph_demod = normalize(de_emphasis(fm_demodulate(fm_emph_noisy, fs=WORK_FS), WORK_FS))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
plot_spectrum(message, fs=WORK_FS, ax=axes[0], title="Original Audio Spectrum")
plot_spectrum(pre, fs=WORK_FS, ax=axes[1], title="Pre-Emphasized Spectrum")
axes[0].set_xlim(0, 6000)
axes[1].set_xlim(0, 6000)
plot_spectrum(emph_demod, fs=WORK_FS, ax=axes[2], title="Recovered Audio After De-Emphasis")
axes[2].set_xlim(0, 6000)
for ax in axes:
    ax.set_ylim(-100, 5)
plt.tight_layout()

display(Markdown("**FM without pre/de-emphasis**"))
display(audio_player(resample_signal(plain_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))
display(Markdown("**FM with pre/de-emphasis**"))
display(audio_player(resample_signal(emph_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_emphasis(snr_db=10.0):
    fm_plain_noisy, _ = add_awgn(fm_plain, snr_db, seed=42)
    fm_emph_noisy, _ = add_awgn(fm_emph, snr_db, seed=42)
    plain_demod = fm_demodulate(fm_plain_noisy, fs=WORK_FS)
    emph_demod = normalize(de_emphasis(fm_demodulate(fm_emph_noisy, fs=WORK_FS), WORK_FS))
    axes[0].clear()
    axes[1].clear()
    plot_spectrum(plain_demod, fs=WORK_FS, ax=axes[0], title="No Emphasis")
    plot_spectrum(emph_demod, fs=WORK_FS, ax=axes[1], title="With Pre/De-Emphasis")
    for ax in axes:
        ax.set_xlim(0, 6000)
        ax.set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(emph_demod, WORK_FS, PLAY_FS), rate=PLAY_FS)

controls = widgets.interactive(
    update_emphasis,
    snr_db=float_slider(min_value=0, max_value=30, step=1, value=10, description="SNR dB"),
)
display(controls, audio_out)


## Key Takeaway

Pre/de-emphasis does not change the message content. It changes where the system spends its SNR budget, which is why it becomes more audible as the channel gets noisy.